In [4]:
import time
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

In [31]:
URL = "https://sensoryfair.com/product-category/trending/sensory-products-tools/adaptive-clothing/"

DELAY_SECONDS = 1.0

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://sensoryfair.com/",
    "Connection": "keep-alive",
}

In [32]:
def fetch_soup(url, session):
    r = session.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

In [33]:
def get_next_page_url(soup, base_url):
    next_a = soup.select_one("a.next.page-numbers")

    if next_a and next_a.get("href"):
        return urljoin(base_url, next_a["href"])

    return None

In [34]:
def get_product_links(soup, base_url):
    links = set()

    product_divs = soup.select("div.type-product")

    for div in product_divs:
        a = div.select_one('a[href*="/product/"]')
        if not a:
            continue

        full_url = urljoin(base_url, a["href"])
        links.add(full_url)

    return sorted(links)

In [35]:
def collect_all_product_links(start_url):
    all_links = set()

    with requests.Session() as session:
        page_url = start_url
        page_number = 1

        while page_url:
            print(f"Scraping category page {page_number}: {page_url}")

            soup = fetch_soup(page_url, session)

            # 1) Collect product links from this page
            page_links = get_product_links(soup, page_url)
            print(f"  Found {len(page_links)} product links on this page")

            for link in page_links:
                all_links.add(link)

            # 2) Move to the next page
            page_url = get_next_page_url(soup, page_url)

            page_number += 1
            time.sleep(DELAY_SECONDS)

    return sorted(all_links)


In [36]:
def scrape_product_page(product_url, session):
    soup = fetch_soup(product_url, session)

    # Product name
    name_el = soup.select_one("h1.product_title") or soup.select_one("h1")
    product_name = name_el.get_text(strip=True) if name_el else ""

    # Short description 
    short_el = soup.select_one("div.woocommerce-product-details__short-description")
    short_description = short_el.get_text(" ", strip=True) if short_el else ""

    # Full description 
    full_el = soup.select_one("#section-description")  # grab the whole block
    full_description = full_el.get_text(" ", strip=True) if full_el else ""

    # Price (optional)
    price_el = soup.select_one("p.price") or soup.select_one("span.price")
    price = price_el.get_text(" ", strip=True) if price_el else ""

    return {
        "product_name": product_name,
        "product_url": product_url,
        "short_description": short_description,
        "full_description": full_description,
        "price": price,
    }

In [37]:
all_product_links = collect_all_product_links(URL)
test = all_product_links[0:10]
print("\nTOTAL unique product links:", len(all_product_links))
print("First 10 product links:")
for link in all_product_links[:10]:
    print(link)

Scraping category page 1: https://sensoryfair.com/product-category/trending/sensory-products-tools/adaptive-clothing/
  Found 10 product links on this page
Scraping category page 2: https://sensoryfair.com/product-category/trending/sensory-products-tools/adaptive-clothing/page/2/
  Found 10 product links on this page
Scraping category page 3: https://sensoryfair.com/product-category/trending/sensory-products-tools/adaptive-clothing/page/3/
  Found 10 product links on this page
Scraping category page 4: https://sensoryfair.com/product-category/trending/sensory-products-tools/adaptive-clothing/page/4/
  Found 10 product links on this page
Scraping category page 5: https://sensoryfair.com/product-category/trending/sensory-products-tools/adaptive-clothing/page/5/
  Found 10 product links on this page
Scraping category page 6: https://sensoryfair.com/product-category/trending/sensory-products-tools/adaptive-clothing/page/6/
  Found 2 product links on this page

TOTAL unique product links: 5

In [38]:
rows = []

with requests.Session() as session:
    for url in tqdm(all_product_links):
        try:
            rows.append(scrape_product_page(url, session))
        except Exception as e:
            rows.append({
                "product_name": "",
                "product_url": url,
                "short_description": "",
                "full_description": "",
                "price": "",
                "error": str(e)
            })

        time.sleep(DELAY_SECONDS)

df = pd.DataFrame(rows)
df

100%|██████████████████████████████████████████████████████████████████████████████████| 52/52 [01:49<00:00,  2.10s/it]


,product_name,product_url,short_description,full_description,price
0,Abby & Noah Sensory Compression Vest for Women...,https://sensoryfair.com/product/abby-noah-sens...,Therapist-Designed Compression Hug: Our “weara...,Therapist-Designed Compression Hug: Our “weara...,$ 72.65 Original price was: $72.65. $ 42.99 Cu...
1,Snugabye Adaptive Back Zip Convert-A-Foot Slee...,https://sensoryfair.com/product/adaptive-back-...,,This collection was designed with your special...,$ 47.53 Original price was: $47.53. $ 34.95 Cu...
2,"CBObaby Adaptive Bodysuit, Short Sleeve, Front...",https://sensoryfair.com/product/adaptive-bodys...,,Discover comfort and functionality with this i...,$ 24.99 Original price was: $24.99. $ 19.99 Cu...
3,EEOST Adaptive Clothing for Kids Special Needs...,https://sensoryfair.com/product/adaptive-cloth...,,,$ 45.80 Original price was: $45.80. $ 28.99 Cu...
4,Benefit Wear Anti-Strip Jumpsuit Clothing for ...,https://sensoryfair.com/product/benefit-wear-a...,,Are you searching for the perfect nightwear or...,$ 68.85 Original price was: $68.85. $ 51.00 Cu...
5,JOF Berlin Body Zipper For Kids | Blue | Onesi...,https://sensoryfair.com/product/berlin-body-zi...,,,$ 67.93 Original price was: $67.93. $ 44.99 Cu...
6,"Hugsmiling Body Sock Sensory Kids, Medium 27 *...",https://sensoryfair.com/product/body-sock-sens...,Our 4-way stretch sensory body sock is an all ...,Our 4-way stretch sensory body sock is an all ...,$ 29.73 Original price was: $29.73. $ 16.99 Cu...
7,"Hugsmiling Body Sock Sensory Kids, Small 24 * ...",https://sensoryfair.com/product/body-sock-sens...,Our 4-way stretch sensory body sock is an all ...,Our 4-way stretch sensory body sock is an all ...,$ 15.99 Original price was: $15.99. $ 14.99 Cu...
8,DC Comics Justice League 2 Pack Adaptive T-Shi...,https://sensoryfair.com/product/dc-comics-just...,,Your little hero is ready for an exciting comi...,$ 32.38 Original price was: $32.38. $ 19.99 Cu...
9,TescoRu Exclusive Back-Zip Onesie – Designed w...,https://sensoryfair.com/product/exclusive-back...,,*** Helpful Tips *** Our Back-Zip Onesie is de...,$ 38.92 Original price was: $38.92. $ 21.99 Cu...


In [39]:
df.to_csv("sensoryfair_adaptive_clothing.csv", index=False)

In [26]:
bad = df[df["error"].notna()].iloc[0]
bad_url = bad["product_url"]
print("BAD URL:", bad_url)

BAD URL: https://sensoryfair.com/product/outree-double-layer-therapy-swing-with-360-swivel-hanger-healing-relaxing-cuddle-swing-for-kids-and-adults-with-autism-adhd-sensory-processing-disorder/


In [27]:
import requests

test_url = bad_url 

with requests.Session() as session:
    r = session.get(test_url, headers=HEADERS, timeout=30, allow_redirects=True)
    print("Status code:", r.status_code)
    print("Final URL after redirects:", r.url)
    print("Response length:", len(r.text))
    print("First 300 chars of HTML:\n", r.text[:300])

Status code: 200
Final URL after redirects: https://sensoryfair.com/product/outree-double-layer-therapy-swing-with-360-swivel-hanger-healing-relaxing-cuddle-swing-for-kids-and-adults-with-autism-adhd-sensory-processing-disorder/
Response length: 230337
First 300 chars of HTML:
 <!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<!-- feeds & pingback -->
<link rel="profile" href="http://gmpg.org/xfn/11" />
<link rel="pingback" href="https://sensoryfair.com/xmlrpc.php" />
<meta nam


In [8]:
with requests.Session() as session:
    page_url = URL
    page_number = 1

    while page_url:
        print(f"Page {page_number}: {page_url}")

        soup = fetch_soup(page_url, session)
        page_url = get_next_page_url(soup, page_url)

        page_number += 1
        time.sleep(DELAY_SECONDS)

    print("Reached last page.")

Page 1: https://sensoryfair.com/product-category/trending/sensory-products-tools/
Page 2: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/2/
Page 3: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/3/
Page 4: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/4/
Page 5: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/5/
Page 6: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/6/
Page 7: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/7/
Page 8: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/8/
Page 9: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/9/
Page 10: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/10/
Page 11: https://sensoryfair.com/product-category/trending/sensory-products-tools/page/11/
Page 12: https://sensory

KeyboardInterrupt: 

In [11]:
with requests.Session() as session:
    soup = fetch_soup(URL, session)
    links = get_product_links(soup, URL)

    print("Links found:", len(links))
    for link in links:
        print(link)


Links found: 10
https://sensoryfair.com/product/abby-noah-sensory-compression-vest-for-women-men-autism-sensory-products-for-kids-soft-wearable-hug-for-sensory-seeking-adhd-calming-vest-black-adult-medium/
https://sensoryfair.com/product/berlin-body-zipper-for-kids-blue-onesie-bodyshort-with-zipper-adaptive-clothing-for-children-with-special-needs/
https://sensoryfair.com/product/body-sock-sensory-kids-medium-27-47-inch-good-for-kids-height-4858-inch-soft-fabric-with-strong-stitching-and-snap-closures-classic-blue/
https://sensoryfair.com/product/dc-comics-justice-league-2-pack-adaptive-t-shirts-sensory-friendly-little-kid-sizes-4-7-8/
https://sensoryfair.com/product/kaycey-special-needs-adaptive-clothing-for-children-long-sleeve-short-leg-zip-back-jumpsuit/
https://sensoryfair.com/product/kids-adaptive-anti-strip-back-zip-lightweight-romper-made-with-100-organic-cotton-designed-for-special-needs/
https://sensoryfair.com/product/outree-sensory-sock-for-kids-small3-5-years39x27good-for-